# Barrido Comparativo de Cuantización para Trazas de Activación (`CompactTrace`) — Colab GPU

Evalúa sobre activaciones reales de `Qwen/Qwen3.5-0.8B` (secuencia GSM8K con razonamiento + loop) la frontera de **tamaño vs fidelidad** de todas las nuevas opciones de `wsi.tracer`:

1. **`uint16` para índices MLP Top-K** (`< 65,535` neuronas): reduce a la mitad el almacenamiento de los índices Top-K (que representaban ~70% de los 85.3 MB anteriores).
2. **Esquemas de cuantización de residuales:**
   - `fp16` (referencia)
   - `int8` (`group_size=16` y `64`)
   - `int4` (`group_size=16` y `64`)
   - `fp4` E2M1 estilo **`NVFP4`** (`{±0, 0.5, 1, 1.5, 2, 3, 4, 6}`, 2 valores/byte)
   - `ternary` óptimo en L2 **1.58-bit** (`{-1, 0, +1}`, empaquetado 5 trits/byte estilo `codehygiene` + escala `fp16`)
3. **Rotación ortogonal Walsh-Hadamard** (`hadamard_block=0` vs `64`) para dispersar outliers antes de cuantizar.
4. **Métricas reales (sin corpus previo, ~1–2 min en L4/A100):**
   - Tamaño total en MB, compresión vs `trace_all` `fp32`, bits efectivos por elemento y bytes ahorrados con `uint16`.
   - Error capa por capa (`mean_cosine`, `min_layer_cos`, `mean_rel_l2`).
   - Fidelidad downstream en `norm -> lm_head`: **KL divergence `KL(P_fp32 || P_quant)`**, **`top1_match`** y **`top5_overlap`**.
   - Preservación funcional de **`detect_loop_onset`**.


In [ ]:
import os, shutil, json
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/wsi_runs/trace_quant_sweep"
OUT = "/content/results_quant_sweep.json"
os.makedirs(DRIVE, exist_ok=True)


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf /content/wsi-repo
!git clone --depth 1 -q https://github.com/krexito/weight-space-index.git /content/wsi-repo
%cd /content/wsi-repo
!pip install -q safetensors transformers accelerate


In [ ]:
%%writefile /content/wsi-repo/wsi/tracer.py
"""Token-level dense & compact activation tracing (`trace_all` forensics + block-quantized `CompactTrace`).

Pairs `wsi.capture.trace_all` with `WeightSpaceIndex` geometry when global averages fail:
  1. `diff_traces(model, ids_a, ids_b, wsi=...)`: full-fidelity X-ray via `trace_all` across two prompts
     (e.g., neutral vs. pushback turn, clean vs. trigger prompt). Handles unequal prompt lengths and
     ranks exact layers, attention heads, MLP neurons (`Δact_i * ||W_down[:, i]||`), cache state shifts,
     attention redistribution, and vocabulary readout (`explain_vector`).
  2. `capture_compact(model, ids, ...)`: scalable token-by-token tracing using micro-block INT8 / INT4 /
     E2M1 FP4 (`NVFP4`-style) / optimal L2 1.58-bit Ternary quantization (with optional Walsh-Hadamard
     rotation to suppress outliers) + Top-K sparse contribution-weighted MLP neurons + per-head L2 energy.
"""

from __future__ import annotations

import json
import re
from dataclasses import dataclass, field
from functools import lru_cache
from pathlib import Path
from typing import Any

import torch
import torch.nn as nn

from .capture import _is_linear, find_blocks, find_mlps, trace_all

_E2M1_POS = (0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0)
_POW3 = (1, 3, 9, 27, 81)


# ----------------------------------------------------------------------------------------------- Hadamard + block quantization
@lru_cache(maxsize=16)
def _hadamard_matrix(n: int) -> torch.Tensor:
    """Normalized n x n Sylvester Hadamard matrix (H @ H.T == I, symmetric orthogonal)."""
    if n < 1 or (n & (n - 1)) != 0:
        raise ValueError(f"Hadamard block size must be power of 2, got {n}")
    h = torch.ones((1, 1), dtype=torch.float32)
    while h.shape[0] < n:
        h = torch.cat([torch.cat([h, h], dim=1), torch.cat([h, -h], dim=1)], dim=0)
    return h / (n ** 0.5)


def resolve_hadamard_block(dim: int, requested: int) -> int:
    """Largest power-of-2 <= requested that divides `dim` (0 disables Hadamard rotation)."""
    if requested <= 0:
        return 0
    b = 1 << (requested.bit_length() - 1)
    while b > 1 and (dim % b) != 0:
        b //= 2
    return b if b > 1 else 0


def hadamard_transform(x: torch.Tensor, block: int) -> torch.Tensor:
    """Orthonormal block Walsh-Hadamard transform on the last axis (inverse == forward)."""
    if block <= 1 or (x.shape[-1] % block) != 0:
        return x.float()
    h = _hadamard_matrix(block).to(device=x.device, dtype=torch.float32)
    orig = x.shape
    return (x.float().reshape(-1, block) @ h).reshape(orig)


def _pad_to_blocks(x: torch.Tensor, group_size: int) -> tuple[torch.Tensor, tuple[int, ...]]:
    orig_shape = tuple(x.shape)
    flat = x.detach().float().reshape(-1)
    pad = (-flat.numel()) % group_size
    if pad:
        flat = torch.cat([flat, torch.zeros(pad, device=flat.device, dtype=torch.float32)])
    return flat.reshape(-1, group_size), orig_shape


def _unpad_shape(flat: torch.Tensor, orig_shape: tuple[int, ...]) -> torch.Tensor:
    n = 1
    for s in orig_shape:
        n *= s
    return flat.reshape(-1)[:n].reshape(orig_shape)


def quantize_block_int4(x: torch.Tensor, group_size: int = 16) -> tuple[torch.Tensor, torch.Tensor, tuple[int, ...]]:
    """Micro-block symmetric INT4 quantization (signed nibbles [-7..7] packed 2/byte + fp16 scale)."""
    if group_size % 2 != 0:
        raise ValueError("group_size must be even for 2-nibble uint8 packing")
    blocks, orig_shape = _pad_to_blocks(x, group_size)
    scales = (blocks.abs().amax(dim=1, keepdim=True) / 7.0).clamp_min(1e-8)
    q = torch.round(blocks / scales).clamp(-7, 7).to(torch.int8) + 8
    q_u8 = q.to(torch.uint8).reshape(-1, 2)
    packed = ((q_u8[:, 0] & 0x0F) | ((q_u8[:, 1] & 0x0F) << 4)).cpu()
    return packed, scales.squeeze(1).to(torch.float16).cpu(), orig_shape


def dequantize_block_int4(packed: torch.Tensor, scales: torch.Tensor, orig_shape: tuple[int, ...],
                          group_size: int = 16) -> torch.Tensor:
    lo = (packed & 0x0F).to(torch.int8) - 8
    hi = ((packed >> 4) & 0x0F).to(torch.int8) - 8
    q = torch.stack([lo, hi], dim=1).reshape(-1, group_size).float()
    return _unpad_shape(q * scales.float().unsqueeze(1), orig_shape)


def quantize_block_fp4(x: torch.Tensor, group_size: int = 16) -> tuple[torch.Tensor, torch.Tensor, tuple[int, ...]]:
    """Micro-block E2M1 FP4 quantization (`NVFP4`-style grid +-{0, 0.5, 1, 1.5, 2, 3, 4, 6}, 2/byte + fp16 scale)."""
    if group_size % 2 != 0:
        raise ValueError("group_size must be even for 2-nibble uint8 packing")
    blocks, orig_shape = _pad_to_blocks(x, group_size)
    grid = torch.tensor(_E2M1_POS, device=blocks.device, dtype=torch.float32)
    scales = (blocks.abs().amax(dim=1, keepdim=True) / 6.0).clamp_min(1e-8)
    normed = blocks / scales
    mags = normed.abs().unsqueeze(-1)
    idx = (mags - grid).abs().argmin(dim=-1).to(torch.uint8)  # 0..7
    sign = (normed < 0).to(torch.uint8) << 3
    nib = (sign | idx).reshape(-1, 2)
    packed = (nib[:, 0] | (nib[:, 1] << 4)).cpu()
    return packed, scales.squeeze(1).to(torch.float16).cpu(), orig_shape


def dequantize_block_fp4(packed: torch.Tensor, scales: torch.Tensor, orig_shape: tuple[int, ...],
                         group_size: int = 16) -> torch.Tensor:
    grid = torch.tensor(_E2M1_POS, device=packed.device, dtype=torch.float32)
    nibs = torch.stack([packed & 0x0F, (packed >> 4) & 0x0F], dim=1).reshape(-1, group_size)
    vals = grid[(nibs & 0x07).long()] * torch.where((nibs & 0x08) != 0, -1.0, 1.0)
    return _unpad_shape(vals * scales.float().unsqueeze(1), orig_shape)


def quantize_block_ternary(x: torch.Tensor, group_size: int = 16) -> tuple[torch.Tensor, torch.Tensor, tuple[int, ...]]:
    """L2-optimal per-block 1.58-bit ternary ({-1, 0, +1}) packed at 5 trits per uint8 byte + fp16 block scale."""
    blocks, orig_shape = _pad_to_blocks(x, group_size)
    abs_b = blocks.abs()
    order = abs_b.argsort(dim=1, descending=True)
    sorted_abs = abs_b.gather(1, order)
    csum = sorted_abs.cumsum(dim=1)
    ks = torch.arange(1, group_size + 1, device=blocks.device, dtype=torch.float32)[None, :]
    best_idx = (csum * csum / ks).argmax(dim=1)  # exact global optimum per block over top-m subsets
    best_k = (best_idx + 1).float()
    best_csum = csum.gather(1, best_idx.unsqueeze(1)).squeeze(1)
    scales = (best_csum / best_k).clamp_min(1e-8)

    # Set non-zero ternary sign on the winning top-m elements of each block
    rank_in_block = order.argsort(dim=1)
    active = rank_in_block <= best_idx.unsqueeze(1)
    trits = torch.where(active, torch.where(blocks >= 0, 2, 0), 1).to(torch.uint8).reshape(-1)  # 0=-1, 1=0, 2=+1

    pad5 = (-trits.numel()) % 5
    if pad5:
        trits = torch.cat([trits, torch.ones(pad5, device=trits.device, dtype=torch.uint8)])
    pow3 = torch.tensor(_POW3, device=trits.device, dtype=torch.uint8)
    packed = (trits.reshape(-1, 5) * pow3).sum(dim=1).to(torch.uint8).cpu()
    return packed, scales.to(torch.float16).cpu(), orig_shape


def dequantize_block_ternary(packed: torch.Tensor, scales: torch.Tensor, orig_shape: tuple[int, ...],
                             group_size: int = 16) -> torch.Tensor:
    n_elems = scales.numel() * group_size
    p3 = torch.tensor(_POW3, device=packed.device, dtype=torch.int32)[None, :]
    trits = ((packed.to(torch.int32).unsqueeze(1) // p3) % 3).reshape(-1)[:n_elems]
    vals = (trits - 1).float().reshape(-1, group_size)
    return _unpad_shape(vals * scales.float().unsqueeze(1), orig_shape)


def quantize_fp8_or_int8(x: torch.Tensor, group_size: int = 16) -> tuple[torch.Tensor, torch.Tensor | None, str]:
    """Quantize to 1 byte/element (micro-block INT8 for portability; scale isolates outliers)."""
    blocks, _ = _pad_to_blocks(x, group_size)
    scales = (blocks.abs().amax(dim=1, keepdim=True) / 127.0).clamp_min(1e-8)
    q = torch.round(blocks / scales).clamp(-127, 127).to(torch.int8).cpu()
    return q.reshape(-1), scales.squeeze(1).to(torch.float16).cpu(), "int8_blk"


def dequantize_fp8_or_int8(q: torch.Tensor, scales: torch.Tensor, orig_shape: tuple[int, ...],
                           group_size: int = 16) -> torch.Tensor:
    deq = q.reshape(-1, group_size).float() * scales.float().unsqueeze(1)
    return _unpad_shape(deq, orig_shape)


def quantization_error(x: torch.Tensor, x_hat: torch.Tensor) -> dict[str, float]:
    xf, hf = x.detach().float().cpu().reshape(-1), x_hat.detach().float().cpu().reshape(-1)
    diff = (xf - hf).norm().item()
    norm = max(xf.norm().item(), 1e-12)
    cos = float(torch.nn.functional.cosine_similarity(xf.unsqueeze(0), hf.unsqueeze(0)).item()) if norm > 1e-12 else 1.0
    return {"rel_l2": round(diff / norm, 6), "cosine": round(cos, 6)}


def _encode_resid(x: torch.Tensor, quant: str, group_size: int, had_block: int) -> dict[str, Any]:
    shp = list(x.shape)
    hb = resolve_hadamard_block(x.shape[-1], had_block)
    xw = hadamard_transform(x, hb) if hb > 1 else x
    if quant == "int4":
        d, s, _ = quantize_block_int4(xw, group_size)
        return {"data": d, "scales": s, "shape": shp}
    if quant == "fp4":
        d, s, _ = quantize_block_fp4(xw, group_size)
        return {"data": d, "scales": s, "shape": shp}
    if quant == "ternary":
        d, s, _ = quantize_block_ternary(xw, group_size)
        return {"data": d, "scales": s, "shape": shp}
    if quant == "int8":
        q, s, _ = quantize_fp8_or_int8(xw, group_size)
        return {"data": q, "scales": s, "shape": shp}
    return {"data": xw.to(torch.float16).cpu(), "scales": None, "shape": shp}


def _decode_resid(r: dict[str, Any], quant: str, group_size: int, had_block: int) -> torch.Tensor:
    shp = tuple(r["shape"])
    if quant == "int4":
        x = dequantize_block_int4(r["data"], r["scales"], shp, group_size)
    elif quant == "fp4":
        x = dequantize_block_fp4(r["data"], r["scales"], shp, group_size)
    elif quant == "ternary":
        x = dequantize_block_ternary(r["data"], r["scales"], shp, group_size)
    elif quant == "int8":
        x = dequantize_fp8_or_int8(r["data"], r["scales"], shp, group_size)
    else:
        x = r["data"].float()
    hb = resolve_hadamard_block(shp[-1], had_block)
    return hadamard_transform(x, hb) if hb > 1 else x


# ----------------------------------------------------------------------------------------------- helpers
def _layer_idx_from_name(name: str) -> int | None:
    m = re.search(r"layers\.(\d+)\.", name)
    if m:
        return int(m.group(1))
    m = re.search(r"\.h\.(\d+)\.", name)
    return int(m.group(1)) if m else None


def _infer_attn_heads(model: nn.Module, out_proj_name: str, d_in: int) -> int:
    mods = dict(model.named_modules())
    parent_name = out_proj_name.rsplit(".", 1)[0] if "." in out_proj_name else ""
    parent = mods.get(parent_name)
    if parent is not None:
        for attr in ("num_v_heads", "linear_num_value_heads", "num_heads", "num_attention_heads", "n_head"):
            v = getattr(parent, attr, None)
            if isinstance(v, int) and v > 0 and d_in % v == 0:
                return v
        for hd_attr in ("head_v_dim", "head_dim", "linear_value_head_dim"):
            hd = getattr(parent, hd_attr, None)
            if isinstance(hd, int) and hd > 0 and d_in % hd == 0:
                return d_in // hd
    cfg = getattr(getattr(model, "config", None), "text_config", getattr(model, "config", None))
    if cfg is not None:
        if "linear_attn" in out_proj_name:
            v = getattr(cfg, "linear_num_value_heads", None)
            if isinstance(v, int) and v > 0 and d_in % v == 0:
                return v
        for attr in ("num_attention_heads", "n_head"):
            v = getattr(cfg, attr, None)
            if isinstance(v, int) and v > 0 and d_in % v == 0:
                return v
    return 1


def _find_attn_out_projs(model: nn.Module, hidden: int) -> dict[str, tuple[nn.Module, int]]:
    out = {}
    for n, m in model.named_modules():
        if "mtp" in n or "visual" in n or not _is_linear(m):
            continue
        if n.endswith((".o_proj", ".out_proj", ".c_proj")) and ".mlp." not in n:
            nh = _infer_attn_heads(model, n, m.weight.shape[1] if isinstance(m, nn.Linear) else m.weight.shape[0])
            out[n] = (m, nh)
    return out


# ----------------------------------------------------------------------------------------------- CompactTrace
@dataclass
class CompactTrace:
    """Scalable token-by-token activation trace combining micro-block quantized residuals, Top-K MLP units,
    and per-head attention energies across full sequences."""

    seq_len: int
    n_layers: int
    hidden_size: int
    resid_quant: str
    group_size: int
    top_k_mlp: int
    resid: dict[int, dict[str, Any]]
    mlp_topk: dict[int, dict[str, torch.Tensor]]
    head_norms: dict[str, dict[str, Any]]
    hadamard_block: int = 0
    meta: dict[str, Any] = field(default_factory=dict)

    def dequant_resid(self, layer: int) -> torch.Tensor:
        return _decode_resid(self.resid[layer], self.resid_quant, self.group_size, self.hadamard_block)

    def size_bytes(self) -> int:
        b = sum(r["data"].numel() * r["data"].element_size()
                + (r["scales"].numel() * r["scales"].element_size() if r["scales"] is not None else 0)
                for r in self.resid.values())
        b += sum(t.numel() * t.element_size() for m in self.mlp_topk.values() for t in m.values())
        b += sum(h["l2"].numel() * h["l2"].element_size() for h in self.head_norms.values())
        return b

    def compare_positions(self, t_target: int, t_ref: int, *, top_k: int = 8) -> dict[str, Any]:
        """Find which layers, MLP neurons, and attention heads shifted strongest at token `t_target` vs `t_ref`."""
        layer_shifts, mlp_shifts, head_shifts = [], [], []
        for l in sorted(self.resid):
            R = self.dequant_resid(l)
            va, vb = R[t_ref], R[t_target]
            layer_shifts.append({
                "layer": l,
                "delta_norm": round(float((vb - va).norm()), 4),
                "cosine": round(float(torch.nn.functional.cosine_similarity(va[None], vb[None])), 5),
            })
        for l, m in sorted(self.mlp_topk.items()):
            ia, va = m["idx"][t_ref].tolist(), m["val"][t_ref].float().tolist()
            ib, vb = m["idx"][t_target].tolist(), m["val"][t_target].float().tolist()
            ma, mb = dict(zip(ia, va)), dict(zip(ib, vb))
            for u in set(ma) | set(mb):
                da, db = ma.get(u, 0.0), mb.get(u, 0.0)
                mlp_shifts.append({"layer": l, "unit": int(u), "ref_contrib": round(da, 4),
                                   "target_contrib": round(db, 4), "delta": round(db - da, 4),
                                   "abs_delta": round(abs(db - da), 4)})
        for mod, h in self.head_norms.items():
            da = h["l2"][t_ref].float()
            db = h["l2"][t_target].float()
            diff = db - da
            for hid in diff.abs().topk(min(3, len(diff))).indices.tolist():
                head_shifts.append({"module": mod, "layer": h["layer"], "head": hid,
                                    "ref_l2": round(float(da[hid]), 4), "target_l2": round(float(db[hid]), 4),
                                    "delta": round(float(diff[hid]), 4), "abs_delta": round(float(diff[hid].abs()), 4)})
        mlp_shifts.sort(key=lambda x: -x["abs_delta"])
        head_shifts.sort(key=lambda x: -x["abs_delta"])
        return {
            "t_target": t_target, "t_ref": t_ref,
            "layer_shifts": sorted(layer_shifts, key=lambda x: -x["delta_norm"])[:top_k],
            "top_mlp_shifts": mlp_shifts[:top_k],
            "top_head_shifts": head_shifts[:top_k],
        }

    def detect_loop_onset(self, layer: int | None = None, window: int = 8, thr: float = 0.96) -> dict[str, Any]:
        """Scan token-to-previous-token residual cosine inside one trace to locate repetition onset `t*`."""
        l = (max(self.resid) // 2) if layer is None else layer
        R = self.dequant_resid(l)
        Rn = R / R.norm(dim=1, keepdim=True).clamp_min(1e-8)
        T = Rn.shape[0]
        if T <= window:
            return {"layer": l, "onset_token": None, "max_lag_cos": 0.0}
        lag_cos = torch.zeros(T)
        for t in range(1, T):
            prev = Rn[max(0, t - window) : t]
            lag_cos[t] = (prev @ Rn[t]).max()
        hits = (lag_cos > thr).nonzero(as_tuple=True)[0].tolist()
        return {
            "layer": l,
            "onset_token": int(hits[0]) if hits else None,
            "max_lag_cos": round(float(lag_cos.max()), 5),
            "mean_tail_cos": round(float(lag_cos[T // 2:].mean()), 5) if T >= 4 else round(float(lag_cos.mean()), 5),
        }

    def save(self, out_dir: str | Path) -> Path:
        from safetensors.torch import save_file

        p = Path(out_dir)
        p.mkdir(parents=True, exist_ok=True)
        flat: dict[str, torch.Tensor] = {}
        resid_shapes = {}
        for l, r in self.resid.items():
            flat[f"resid.{l}.data"] = r["data"].contiguous()
            if r["scales"] is not None:
                flat[f"resid.{l}.scales"] = r["scales"].contiguous()
            resid_shapes[str(l)] = list(r["shape"])
        for l, m in self.mlp_topk.items():
            for k, v in m.items():
                flat[f"mlp.{l}.{k}"] = v.contiguous()
        head_meta = {}
        for idx, (mod, h) in enumerate(self.head_norms.items()):
            flat[f"head.{idx}"] = h["l2"].contiguous()
            head_meta[str(idx)] = {"module": mod, "layer": h["layer"], "n_heads": h["n_heads"]}
        save_file(flat, str(p / "compact_trace.safetensors"))
        (p / "compact_trace.json").write_text(json.dumps({
            "seq_len": self.seq_len, "n_layers": self.n_layers, "hidden_size": self.hidden_size,
            "resid_quant": self.resid_quant, "group_size": self.group_size, "top_k_mlp": self.top_k_mlp,
            "hadamard_block": self.hadamard_block, "resid_shapes": resid_shapes, "heads": head_meta,
            "meta": self.meta,
        }, indent=2), encoding="utf-8")
        return p

    @classmethod
    def load(cls, trace_dir: str | Path) -> "CompactTrace":
        from safetensors.torch import load_file

        p = Path(trace_dir)
        info = json.loads((p / "compact_trace.json").read_text(encoding="utf-8"))
        tens = load_file(str(p / "compact_trace.safetensors"))
        resid, mlp_topk, head_norms = {}, {}, {}
        for l_str, shp in info["resid_shapes"].items():
            l = int(l_str)
            resid[l] = {"data": tens[f"resid.{l}.data"], "scales": tens.get(f"resid.{l}.scales"), "shape": shp}
        for l in range(info["n_layers"]):
            if f"mlp.{l}.idx" in tens:
                mlp_topk[l] = {"idx": tens[f"mlp.{l}.idx"], "val": tens[f"mlp.{l}.val"],
                               "energy_frac": tens[f"mlp.{l}.energy_frac"]}
        for idx, hm in info["heads"].items():
            head_norms[hm["module"]] = {"layer": hm["layer"], "n_heads": hm["n_heads"], "l2": tens[f"head.{idx}"]}
        return cls(
            info["seq_len"], info["n_layers"], info["hidden_size"], info["resid_quant"],
            info["group_size"], info["top_k_mlp"], resid, mlp_topk, head_norms,
            hadamard_block=info.get("hadamard_block", 0), meta=info.get("meta", {}),
        )


@torch.no_grad()
def capture_compact(
    model: nn.Module,
    input_ids: torch.Tensor | list[int],
    *,
    top_k_mlp: int = 256,
    resid_quant: str = "int8",
    group_size: int = 16,
    hadamard_block: int = 0,
    max_tokens: int | None = None,
) -> CompactTrace:
    """Record a compressed token-by-token trace: quantized residuals, Top-K weighted MLP units, and head L2 norms."""
    dev = next(model.parameters()).device
    ids = (input_ids if isinstance(input_ids, torch.Tensor) else torch.tensor(input_ids)).reshape(1, -1)
    if max_tokens is not None:
        ids = ids[:, :max_tokens]
    T = ids.shape[1]

    _, blocks = find_blocks(model)
    n_layers = len(blocks)
    mods = dict(model.named_modules())
    mlps = find_mlps(model)
    down_norms = {}
    mlp_by_down = {}
    for g in mlps:
        m = mods[g.down]
        w = m.weight.detach().float()
        down_norms[g.down] = w.norm(dim=0) if isinstance(m, nn.Linear) else w.norm(dim=1)
        li = _layer_idx_from_name(g.down)
        mlp_by_down[g.down] = li if li is not None else len(mlp_by_down)

    d0 = mods[mlps[0].down]
    hidden = d0.weight.shape[0] if isinstance(d0, nn.Linear) else d0.weight.shape[1]
    attns = _find_attn_out_projs(model, hidden)

    resid_out: dict[int, dict[str, Any]] = {}
    mlp_topk_out: dict[int, dict[str, torch.Tensor]] = {}
    head_out: dict[str, dict[str, Any]] = {}
    handles = []

    for l, blk in enumerate(blocks):
        def bhook(mod, inp, out, l=l):
            t = out[0] if isinstance(out, tuple) else out
            if not isinstance(t, torch.Tensor):
                return
            x = t.detach().reshape(-1, t.shape[-1])
            resid_out[l] = _encode_resid(x, resid_quant, group_size, hadamard_block)

        handles.append(blk.register_forward_hook(bhook))

    for dname, li in mlp_by_down.items():
        m = mods[dname]
        wn = down_norms[dname]

        def mhook(mod, inp, out, li=li, wn=wn):
            x = inp[0].detach().reshape(-1, inp[0].shape[-1]).float()
            idx_dt = torch.uint16 if x.shape[1] <= 65_535 else torch.int32
            contrib = x * wn.to(x.device)
            k = min(top_k_mlp, contrib.shape[1])
            _, idx = contrib.abs().topk(k, dim=1)
            signed = contrib.gather(1, idx)
            tot_sq = (contrib * contrib).sum(dim=1).clamp_min(1e-12)
            kept_sq = (signed * signed).sum(dim=1)
            mlp_topk_out[li] = {
                "idx": idx.to(idx_dt).cpu(),
                "val": signed.to(torch.float16).cpu(),
                "energy_frac": (kept_sq / tot_sq).sqrt().to(torch.float16).cpu(),
            }

        handles.append(m.register_forward_hook(mhook))

    for aname, (am, nh) in attns.items():
        li = _layer_idx_from_name(aname)

        def ahook(mod, inp, out, aname=aname, li=li, nh=nh):
            x = inp[0].detach().reshape(-1, inp[0].shape[-1]).float()
            l2 = x.reshape(x.shape[0], nh, -1).norm(dim=2)
            head_out[aname] = {"layer": li, "n_heads": nh, "l2": l2.to(torch.float16).cpu()}

        handles.append(am.register_forward_hook(ahook))

    try:
        model(input_ids=ids.to(dev), use_cache=False)
    finally:
        for h in handles:
            h.remove()

    return CompactTrace(T, n_layers, hidden, resid_quant, group_size, top_k_mlp,
                        resid_out, mlp_topk_out, head_out, hadamard_block=hadamard_block)


# ----------------------------------------------------------------------------------------------- dense trace_all diff
@torch.no_grad()
def diff_traces(
    model: nn.Module,
    ids_a: torch.Tensor | list[int],
    ids_b: torch.Tensor | list[int],
    *,
    wsi: Any | None = None,
    pos_a: int = -1,
    pos_b: int = -1,
    max_tokens: int = 64,
    top_k: int = 10,
) -> dict[str, Any]:
    """Full-fidelity X-ray comparing trace_all(ids_a) vs trace_all(ids_b) and linking shifts to WSI weights."""
    tr_a = trace_all(model, ids_a, max_tokens=max_tokens)
    tr_b = trace_all(model, ids_b, max_tokens=max_tokens)
    mods = dict(model.named_modules())

    def _at_pos(t: torch.Tensor, pos: int) -> torch.Tensor:
        return (t[0, pos] if t.dim() == 3 else t[pos]).float()

    # 1. Layer-by-layer residual curve across hidden_states
    hs_layers = sorted(
        int(k.split(".")[1]) for k in tr_a if k.startswith("hidden_states.") and k in tr_b
    )
    layer_curve = []
    peak_layer, peak_vec, peak_norm = None, None, -1.0
    for l in hs_layers:
        va = _at_pos(tr_a[f"hidden_states.{l}"], pos_a)
        vb = _at_pos(tr_b[f"hidden_states.{l}"], pos_b)
        dv = vb - va
        dn = float(dv.norm())
        rel = dn / max(float(va.norm()), 1e-12)
        cos = float(torch.nn.functional.cosine_similarity(va[None], vb[None]))
        layer_curve.append({"layer": l, "delta_norm": round(dn, 4), "rel_delta": round(rel, 5), "cosine": round(cos, 5)})
        if l > 0 and dn > peak_norm:
            peak_layer, peak_vec, peak_norm = l, dv, dn

    # 2. Causal MLP neuron shifts: delta_in[i] * ||W_down[:, i]|| (only call WSI explain on the final top_k)
    mlps = find_mlps(model)
    mlp_hits = []
    for g in mlps:
        key = f"{g.down}:in"
        if key not in tr_a or key not in tr_b:
            continue
        da = _at_pos(tr_a[key], pos_a)
        db = _at_pos(tr_b[key], pos_b)
        delta = db - da
        m = mods[g.down]
        w = m.weight.detach().float().cpu()
        wn = w.norm(dim=0) if isinstance(m, nn.Linear) else w.norm(dim=1)
        impact = delta * wn
        li = _layer_idx_from_name(g.down)
        for uid in impact.abs().topk(min(top_k, len(impact))).indices.tolist():
            mlp_hits.append({
                "layer": li, "module": g.down, "unit": int(uid),
                "act_a": round(float(da[uid]), 4), "act_b": round(float(db[uid]), 4),
                "resid_shift_norm": round(float(impact[uid].abs()), 4),
                "signed_shift": round(float(impact[uid]), 4),
            })
    mlp_hits.sort(key=lambda r: -r["resid_shift_norm"])
    top_mlp = mlp_hits[:top_k]
    if wsi is not None:
        for row in top_mlp:
            if row["layer"] is not None:
                try:
                    row["transduction_rule"] = wsi.explain_neuron(layer=row["layer"], unit=row["unit"])["transduction_rule"]
                except Exception:
                    pass

    # 3. Exact Attention / Linear-Attn Head output vector shift ||(x_b - x_a)_h @ W_O_h^T||
    hidden = peak_vec.numel() if peak_vec is not None else 0
    attns = _find_attn_out_projs(model, hidden) if hidden else {}
    head_hits = []
    for aname, (am, nh) in attns.items():
        key = f"{aname}:in"
        if key not in tr_a or key not in tr_b:
            continue
        da = _at_pos(tr_a[key], pos_a)
        db = _at_pos(tr_b[key], pos_b)
        dx = (db - da).reshape(nh, -1)
        w = am.weight.detach().float().cpu()
        w_heads = w.t().reshape(nh, -1, w.shape[0]) if isinstance(am, nn.Linear) else w.reshape(nh, -1, w.shape[1])
        out_deltas = torch.einsum("hd,hdo->ho", dx, w_heads)
        norms = out_deltas.norm(dim=1)
        li = _layer_idx_from_name(aname)
        for hid in norms.topk(min(3, nh)).indices.tolist():
            row = {"layer": li, "module": aname, "head": int(hid), "out_delta_norm": round(float(norms[hid]), 4)}
            if li is not None and f"attentions.{li}" in tr_a and f"attentions.{li}" in tr_b:
                tb_att, ta_att = tr_b[f"attentions.{li}"], tr_a[f"attentions.{li}"]
                if tb_att.dim() == 4 and hid < tb_att.shape[1]:
                    att_a = ta_att[0, hid, pos_a].float()
                    att_b = tb_att[0, hid, pos_b].float()
                    row["attn_a_top_src_pos"] = int(att_a.argmax())
                    row["attn_a_top_weight"] = round(float(att_a.max()), 4)
                    row["attn_b_top_src_pos"] = int(att_b.argmax())
                    row["attn_b_top_weight"] = round(float(att_b.max()), 4)
                    shared = min(len(att_a), len(att_b))
                    if len(att_a) == len(att_b):
                        row["attn_max_shift_pos"] = int((att_b - att_a).abs().argmax())
                        row["attn_max_shift"] = round(float((att_b - att_a).abs().max()), 4)
                    elif shared > 0:
                        row["attn_prefix_l1_shift"] = round(float((att_b[:shared] - att_a[:shared]).abs().sum()), 4)
            head_hits.append(row)
    head_hits.sort(key=lambda r: -r["out_delta_norm"])

    # 4. Cache state deltas (KV at pos_a/pos_b even when T differs, + Gated DeltaNet recurrent/conv states)
    cache_shifts = []
    for k in sorted(set(tr_a) & set(tr_b)):
        if not k.startswith("cache."):
            continue
        ta, tb = tr_a[k].float(), tr_b[k].float()
        if ta.shape == tb.shape:
            cache_shifts.append({"key": k, "delta_norm": round(float((tb - ta).norm()), 4)})
        elif ta.dim() == 4 and tb.dim() == 4 and ta.shape[:2] == tb.shape[:2] and ta.shape[3] == tb.shape[3]:
            va, vb = ta[:, :, pos_a, :], tb[:, :, pos_b, :]
            cache_shifts.append({"key": f"{k}[pos]", "delta_norm": round(float((vb - va).norm()), 4)})
    cache_shifts.sort(key=lambda r: -r["delta_norm"])

    # 5. Optional static readout of peak residual shift via WSI
    vec_readout = None
    if wsi is not None and peak_vec is not None and peak_norm > 1e-6:
        try:
            ex = wsi.explain_vector(peak_vec / peak_vec.norm(), top_k=6)
            vec_readout = {"layer": peak_layer, "promoted": ex.get("promoted_tokens", []),
                           "suppressed": ex.get("suppressed_tokens", [])}
        except Exception:
            pass

    return {
        "len_a": int(tr_a[f"hidden_states.{hs_layers[0]}"].shape[1]) if hs_layers else None,
        "len_b": int(tr_b[f"hidden_states.{hs_layers[0]}"].shape[1]) if hs_layers else None,
        "pos_a": pos_a, "pos_b": pos_b,
        "peak_residual_layer": peak_layer, "peak_residual_delta_norm": round(peak_norm, 4),
        "layer_curve": layer_curve,
        "top_mlp_contributors": top_mlp,
        "top_attention_heads": head_hits[:top_k],
        "top_cache_deltas": cache_shifts[:top_k],
        "peak_vector_readout": vec_readout,
    }


In [ ]:
%%writefile /content/wsi-repo/wsi/__init__.py
"""Weight-Space Index (WSI): Semantic and structural indexing of neural network weights for LLMs.

Built to work standalone or directly on top of `wsr` (Weight-Space Reading and Programming).
"""
from __future__ import annotations

from .capture import ActivationIndex, SelectiveMoI, TaskRouter, apply_selection, capture_index, find_mlps, trace_all
from .compare import (
    AdditiveMoI,
    DynamicCircuitPager,
    MixtureOfIndexes,
    align_semantic_circuits,
    decompile_edit_intent,
    diff_checkpoints,
    discover_connected_chains,
    transfer_edit,
    transfer_transduction_edit,
    transplant_compositional_and_chain,
    transplant_connected_chain,
)
from .indexer import build_index
from .moipatch import Edit, MoIPatch, PatchHandle
from .query import WeightSpaceIndex, load_index
from .tracer import (
    CompactTrace,
    capture_compact,
    dequantize_block_fp4,
    dequantize_block_int4,
    dequantize_block_ternary,
    diff_traces,
    hadamard_transform,
    quantization_error,
    quantize_block_fp4,
    quantize_block_int4,
    quantize_block_ternary,
)

__all__ = [
    "ActivationIndex",
    "AdditiveMoI",
    "CompactTrace",
    "Edit",
    "MoIPatch",
    "PatchHandle",
    "SelectiveMoI",
    "TaskRouter",
    "apply_selection",
    "capture_compact",
    "capture_index",
    "dequantize_block_fp4",
    "dequantize_block_int4",
    "dequantize_block_ternary",
    "diff_traces",
    "find_mlps",
    "hadamard_transform",
    "quantization_error",
    "quantize_block_fp4",
    "quantize_block_int4",
    "quantize_block_ternary",
    "trace_all",
    "DynamicCircuitPager",
    "MixtureOfIndexes",
    "WeightSpaceIndex",
    "align_semantic_circuits",
    "build_index",
    "decompile_edit_intent",
    "diff_checkpoints",
    "discover_connected_chains",
    "load_index",
    "transfer_edit",
    "transfer_transduction_edit",
    "transplant_compositional_and_chain",
    "transplant_connected_chain",
]
__version__ = "0.1.0"


In [ ]:
%%writefile /content/wsi-repo/scripts/experiment_quant_sweep.py
"""Quantization sweep for WSI token activation traces (`CompactTrace` / `trace_all`).

Sweeps INT8, INT4, NVFP4-style E2M1 FP4, and L2-optimal 1.58-bit Ternary (5 trits/byte)
with optional Walsh-Hadamard block rotation across group sizes on real Qwen3.5 hidden states.

Measures per config:
  - Exact compressed CompactTrace size (bytes, MB, bits/elem, compression vs dense fp32 trace_all)
    and exact `uint16` vs legacy `int32` Top-K index byte savings.
  - Layer-wise residual reconstruction (`mean_cosine`, `min_cosine`, `mean_rel_l2`, `last_rel_l2`).
  - Downstream `norm -> lm_head` readout fidelity (`KL(P_fp32 || P_quant)`, `top1_match`, `top5_overlap`).
  - Functional diagnostics agreement (`detect_loop_onset` onset token & max lag cosine vs fp16).

Usage (Colab / GPU):
    python scripts/experiment_quant_sweep.py --model Qwen/Qwen3.5-0.8B --out results_quant_sweep.json
"""

from __future__ import annotations

import argparse
import copy
import gc
import json
import sys
import time
from pathlib import Path

import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(ROOT))

from wsi.capture import find_blocks, trace_all  # noqa: E402
from wsi.tracer import (  # noqa: E402
    _decode_resid,
    _encode_resid,
    capture_compact,
    quantization_error,
)

SWEEP_CONFIGS: list[tuple[str, int, int]] = [
    ("fp16", 16, 0),
    ("int8", 16, 0),
    ("int8", 64, 64),
    ("int4", 16, 0),
    ("int4", 16, 64),
    ("int4", 64, 0),
    ("int4", 64, 64),
    ("fp4", 16, 0),
    ("fp4", 16, 64),
    ("fp4", 64, 0),
    ("fp4", 64, 64),
    ("ternary", 8, 0),
    ("ternary", 16, 0),
    ("ternary", 16, 64),
    ("ternary", 32, 0),
    ("ternary", 32, 64),
    ("ternary", 64, 0),
    ("ternary", 64, 64),
]

CANDIDATE_PROMPTS = [
    "Janet's ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?",
    "A robe takes 2 bolts of blue fiber and half that much white fiber. How many bolts in total does it take?",
    "Josh decides to try flipping a house. He buys a house for $80,000 and then puts in $50,000 in repairs. This increased the value of the house by 150%. How much profit did he make?",
    "James decides to run 3 sprints 3 times a week. He runs 60 meters each sprint. How many total meters does he run a week?",
]


def _local_or_hub(name_or_dir: str) -> str:
    p = Path(name_or_dir)
    if p.exists():
        return str(p)
    from huggingface_hub import snapshot_download
    return snapshot_download(name_or_dir)


def _load_model(mdir: str, dev: torch.device):
    dtype = torch.bfloat16 if dev.type == "cuda" else torch.float32
    try:
        return AutoModelForCausalLM.from_pretrained(mdir, dtype=dtype, attn_implementation="eager").to(dev).eval()
    except Exception:
        return AutoModelForCausalLM.from_pretrained(mdir, dtype=dtype).to(dev).eval()


def _find_norm_and_head(model):
    head = model.get_output_embeddings()
    if head is None:
        head = getattr(model, "lm_head", None)
    for owner in (
        getattr(model, "model", None),
        getattr(getattr(model, "model", None), "language_model", None),
        getattr(model, "language_model", None),
        model,
    ):
        if owner is not None and hasattr(owner, "norm"):
            return owner.norm, head
    raise RuntimeError("Could not locate final norm + lm_head on model")


def _trace_byte_breakdown(ct) -> dict[str, int]:
    resid_b = sum(
        r["data"].numel() * r["data"].element_size()
        + (r["scales"].numel() * r["scales"].element_size() if r["scales"] is not None else 0)
        for r in ct.resid.values()
    )
    topk_b = sum(t.numel() * t.element_size() for m in ct.mlp_topk.values() for t in m.values())
    head_b = sum(h["l2"].numel() * h["l2"].element_size() for h in ct.head_norms.values())
    idx_elem_bytes = next(iter(ct.mlp_topk.values()))["idx"].element_size() if ct.mlp_topk else 2
    n_idx = sum(m["idx"].numel() for m in ct.mlp_topk.values())
    uint16_saved = n_idx * max(0, 4 - idx_elem_bytes)
    return {
        "resid_bytes": int(resid_b),
        "mlp_topk_bytes": int(topk_b),
        "head_norms_bytes": int(head_b),
        "total_bytes": int(ct.size_bytes()),
        "uint16_saved_bytes": int(uint16_saved),
    }


def _logit_metrics(log_p_true: torch.Tensor, p_true: torch.Tensor, top1_true: torch.Tensor,
                   top5_true: torch.Tensor, h_last: torch.Tensor, norm_fp32, head_fp32,
                   dev: torch.device) -> dict[str, float]:
    with torch.no_grad():
        logits_q = head_fp32(norm_fp32(h_last.to(dev))).float()
        log_q = F.log_softmax(logits_q, dim=-1)
        kl = float((p_true * (log_p_true - log_q)).sum(dim=-1).mean().item())
        top1_match = float((logits_q.argmax(dim=-1) == top1_true).float().mean().item())
        top5_q = logits_q.topk(5, dim=-1).indices
        overlap = float((top5_q.unsqueeze(-1) == top5_true.unsqueeze(-2)).any(dim=-1).float().mean().item())
    return {"kl_fp32": round(kl, 6), "top1_match": round(top1_match, 4), "top5_overlap": round(overlap, 4)}


def main() -> None:
    ap = argparse.ArgumentParser(description="Quantization sweep for WSI CompactTrace activations")
    ap.add_argument("--model", default="Qwen/Qwen3.5-0.8B")
    ap.add_argument("--device", default="auto", choices=["auto", "cuda", "cpu"])
    ap.add_argument("--max-new-tokens", type=int, default=1536)
    ap.add_argument("--eval-tokens", type=int, default=256)
    ap.add_argument("--top-k-mlp", type=int, default=256)
    ap.add_argument("--out", default="results_quant_sweep.json")
    args = ap.parse_args()

    dev = torch.device("cuda" if (args.device == "auto" and torch.cuda.is_available()) else
                       ("cpu" if args.device == "auto" else args.device))
    print(f"Device: {dev} | Model: {args.model}", flush=True)

    mdir = _local_or_hub(args.model)
    tok = AutoTokenizer.from_pretrained(mdir)
    model = _load_model(mdir, dev)
    _, blocks = find_blocks(model)
    n_layers = len(blocks)
    loop_layer = min(15, n_layers - 1)

    # 1. Pick a real sequence (preferring one with loop onset, like experiment_trace_xray.py)
    seq, prompt_len, gen_s, chosen_q = None, 0, 0.0, CANDIDATE_PROMPTS[0]
    for q in CANDIDATE_PROMPTS:
        prompt = tok.apply_chat_template([{"role": "user", "content": q}], tokenize=False,
                                         add_generation_prompt=True, enable_thinking=True)
        enc = tok(prompt, return_tensors="pt").to(dev)
        t0 = time.perf_counter()
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=args.max_new_tokens, do_sample=False, use_cache=True)
        gen_s = round(time.perf_counter() - t0, 2)
        cand = gen[0].cpu()
        p_len = int(enc["input_ids"].shape[1])
        ct_try = capture_compact(model, cand, top_k_mlp=args.top_k_mlp, resid_quant="int8", group_size=16)
        onset_try = ct_try.detect_loop_onset(layer=loop_layer, window=16, thr=0.96)
        seq, prompt_len, chosen_q = cand, p_len, q
        print(f"Gen {int(cand.numel()) - p_len} tok in {gen_s}s | loop_onset={onset_try}", flush=True)
        if onset_try["onset_token"] is not None:
            break

    T = int(seq.numel())
    # Dense fp32 trace_all reference baseline over <= 32 tokens
    slice32 = seq[: min(32, T)]
    tr32 = trace_all(model, slice32, max_tokens=32)
    dense_fp32_per_tok = sum(v.numel() * 4 for v in tr32.values()) / max(1, int(slice32.numel()))
    dense_fp32_full_bytes = int(dense_fp32_per_tok * T)
    del tr32
    gc.collect()

    # Exact fp32 hidden states over strided tokens spanning prompt -> reasoning -> loop tail
    eval_seq = seq[: min(T, 1600)]
    eval_len = int(eval_seq.numel())
    with torch.no_grad():
        hs_all = model(input_ids=eval_seq.unsqueeze(0).to(dev), output_hidden_states=True, use_cache=False).hidden_states
    eval_n = min(args.eval_tokens, eval_len)
    pick_idx = torch.linspace(0, eval_len - 1, steps=eval_n).long()
    true_hs = [hs_all[l + 1][0, pick_idx].float().cpu() for l in range(n_layers)]
    hidden_size = int(true_hs[0].shape[-1])
    del hs_all
    if dev.type == "cuda":
        torch.cuda.empty_cache()

    # Prepare fp32 final norm + lm_head copies for accurate logit KL comparison
    norm_mod, head_mod = _find_norm_and_head(model)
    norm_fp32 = copy.deepcopy(norm_mod).to(dev).float().eval()
    head_fp32 = copy.deepcopy(head_mod).to(dev).float().eval()
    with torch.no_grad():
        logits_true = head_fp32(norm_fp32(true_hs[-1].to(dev))).float()
        log_p_true = F.log_softmax(logits_true, dim=-1)
        p_true = log_p_true.exp()
        top1_true = logits_true.argmax(dim=-1)
        top5_true = logits_true.topk(5, dim=-1).indices

    # Measure model native bfloat16 precision floor on the same last-layer hidden state
    bf16_floor = _logit_metrics(log_p_true, p_true, top1_true, top5_true,
                                true_hs[-1].bfloat16().float(), norm_fp32, head_fp32, dev)

    # Run fp16 CompactTrace reference first for loop onset baseline
    ref_onset = None
    rows: list[dict] = []
    header = f"{'cfg':<20} {'MB':>6} {'vs_fp32':>8} {'b/el':>5} {'cos':>8} {'relL2':>7} {'KL':>8} {'top1':>6} {'onset':>6} {'s':>5}"
    print("\n" + header, flush=True)
    print("-" * len(header), flush=True)

    for quant, gs, hb in SWEEP_CONFIGS:
        label = f"{quant}_g{gs}" + (f"_had{hb}" if hb > 0 else "")
        # 1) Full-sequence capture_compact timing, bytes, and loop onset
        t1 = time.perf_counter()
        ct = capture_compact(
            model, seq, top_k_mlp=args.top_k_mlp,
            resid_quant=quant, group_size=gs, hadamard_block=hb,
        )
        cap_s = round(time.perf_counter() - t1, 2)
        binfo = _trace_byte_breakdown(ct)
        onset = ct.detect_loop_onset(layer=loop_layer, window=16, thr=0.96)
        if quant == "fp16" and ref_onset is None:
            ref_onset = onset

        # 2) Residual fidelity across ALL layers on eval slice + downstream lm_head KL
        layer_cos, layer_l2 = [], []
        h_last_hat = None
        for l in range(n_layers):
            enc_l = _encode_resid(true_hs[l], quant, gs, hb)
            dec_l = _decode_resid(enc_l, quant, gs, hb)
            err = quantization_error(true_hs[l], dec_l)
            layer_cos.append(err["cosine"])
            layer_l2.append(err["rel_l2"])
            if l == n_layers - 1:
                h_last_hat = dec_l
        logit_m = _logit_metrics(log_p_true, p_true, top1_true, top5_true, h_last_hat, norm_fp32, head_fp32, dev)

        bits_per_elem = round(binfo["resid_bytes"] * 8.0 / max(1, n_layers * T * hidden_size), 2)
        comp_vs_dense = round(dense_fp32_full_bytes / max(1, binfo["total_bytes"]), 1)
        mb = round(binfo["total_bytes"] / 2**20, 2)
        mean_cos = round(sum(layer_cos) / len(layer_cos), 6)
        min_cos = round(min(layer_cos), 6)
        mean_l2 = round(sum(layer_l2) / len(layer_l2), 6)
        onset_tok = onset["onset_token"]
        ref_tok = ref_onset["onset_token"] if ref_onset else None
        onset_delta = (onset_tok - ref_tok) if (onset_tok is not None and ref_tok is not None) else None

        row = {
            "config": label,
            "resid_quant": quant,
            "group_size": gs,
            "hadamard_block": hb,
            "capture_s": cap_s,
            **binfo,
            "total_mb": mb,
            "kb_per_tok": round(binfo["total_bytes"] / max(1, T) / 1024, 2),
            "resid_bits_per_elem": bits_per_elem,
            "compression_vs_dense_fp32": comp_vs_dense,
            "mean_cos": mean_cos,
            "min_layer_cos": min_cos,
            "mean_rel_l2": mean_l2,
            "last_layer_cos": layer_cos[-1],
            "last_layer_rel_l2": layer_l2[-1],
            **logit_m,
            "loop_onset_token": onset_tok,
            "loop_onset_delta_vs_fp16": onset_delta,
            "loop_max_lag_cos": onset["max_lag_cos"],
        }
        rows.append(row)
        print(
            f"{label:<20} {mb:>6.2f} {comp_vs_dense:>7.1f}x {bits_per_elem:>5.2f} "
            f"{mean_cos:>8.5f} {mean_l2:>7.4f} {logit_m['kl_fp32']:>8.5f} "
            f"{logit_m['top1_match']:>6.3f} {str(onset_tok):>6} {cap_s:>5.1f}",
            flush=True,
        )
        del ct
        gc.collect()

    del norm_fp32, head_fp32, model
    if dev.type == "cuda":
        torch.cuda.empty_cache()

    out = {
        "model": args.model,
        "device": str(dev),
        "n_layers": n_layers,
        "hidden_size": hidden_size,
        "seq_len": T,
        "prompt_len": prompt_len,
        "eval_tokens": eval_n,
        "top_k_mlp": args.top_k_mlp,
        "chosen_prompt": chosen_q,
        "dense_fp32_full_bytes": dense_fp32_full_bytes,
        "dense_fp32_mb": round(dense_fp32_full_bytes / 2**20, 2),
        "bfloat16_logit_floor": bf16_floor,
        "loop_layer": loop_layer,
        "fp16_ref_onset": ref_onset,
        "results": rows,
    }
    Path(args.out).write_text(json.dumps(out, indent=2), encoding="utf-8")
    print(f"\nSaved quantization sweep results to {args.out}", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
!python -u scripts/experiment_quant_sweep.py --model Qwen/Qwen3.5-0.8B --out {OUT}
shutil.copy2(OUT, f"{DRIVE}/results_quant_sweep.json")
d = json.load(open(OUT))
print(f"\nModel={d['model']} | T={d['seq_len']} tok | dense fp32~={d['dense_fp32_mb']} MB | bf16 floor KL={d['bfloat16_logit_floor']['kl_fp32']} top1={d['bfloat16_logit_floor']['top1_match']}")
print(f"uint16 Top-K idx savings per trace: {d['results'][0]['uint16_saved_bytes']/2**20:.2f} MB")
print(f"\n{'config':<20} {'MB':>6} {'ratio':>7} {'b/el':>5} {'cos':>8} {'KL':>8} {'top1':>6} {'onset':>6}")
for r in d["results"]:
    print(f"{r['config']:<20} {r['total_mb']:>6.2f} {r['compression_vs_dense_fp32']:>6.1f}x {r['resid_bits_per_elem']:>5.2f} {r['mean_cos']:>8.5f} {r['kl_fp32']:>8.5f} {r['top1_match']:>6.3f} {str(r['loop_onset_token']):>6}")
from google.colab import files
files.download(OUT)


In [ ]:
# Apagar el runtime automáticamente si el resultado ya está guardado en Drive
AUTO_APAGAR = True
import time
if AUTO_APAGAR and os.path.exists(f"{DRIVE}/results_quant_sweep.json"):
    time.sleep(60)
    from google.colab import runtime
    runtime.unassign()
